In [2]:
# Auto-switch to repo root so relative paths (configs/, scripts/, data/) work
from pathlib import Path
import os

_cwd = Path.cwd()
_root = _cwd
while _root != _root.parent:
    if (_root / "configs" / "config.yaml").exists():
        break
    _root = _root.parent
os.chdir(_root)
print(f"Repo root: {_root}")


Repo root: c:\Users\karki\Downloads\main_openworld\openworld


# 04 — CLIP region recognition (Phase 4)

Two experiments:
1. **GT boxes + CLIP** — perfect-localization recognition ceiling (no detector needed).
2. **RPN proposals + CLIP** — the real pipeline, with RPN-objectness / CLIP fusion (alpha sweep).


Experiment A: CLIP on ground-truth boxes (perfect localization).

In [2]:
%run scripts/run_clip.py --mode gt --config configs/config.yaml \
    --image-ids data/splits/val_tune_image_ids.json \
    --annotations data/annotations/instances_val_tune2017.stripped.json \
    --images-root data/images/val2017 \
    --expansion 0.2 --out-dir outputs/clip/val_tune_gt


c:\Users\karki\Downloads\main_openworld\openworld\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


C:\Users\karki\Downloads\main_openworld\openworld\src\openworld\clip_regions.py:48: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:214.)
  t = torch.from_numpy(np.asarray(image)).permute(2, 0, 1).float() / 255.0


mode=gt images=500 predictions=2939
saved -> outputs\clip\val_tune_gt\predictions.json


Per-class top-1 accuracy of CLIP on GT regions (predicted class vs ground-truth class).

In [3]:
import json
from collections import defaultdict

preds = json.load(open("outputs/clip/val_tune_gt/predictions.json"))
correct = defaultdict(int)
total = defaultdict(int)
for p in preds:
    gt_cid = p.get("gt_category_id", p["category_id"])
    pred_cid = p["category_id"]
    total[gt_cid] += 1
    if gt_cid == pred_cid:
        correct[gt_cid] += 1

sel = json.load(open("data/splits/selected_categories.json"))
orig_to_name = {c["id"]: c["name"] for c in sel["base"] + sel["novel"]}
print(f"{'class':14s} {'GT boxes':>8s} {'top-1 acc':>8s}")
for cid in sorted(total):
    print(f"{orig_to_name.get(cid, cid):14s} {total[cid]:8d} {correct[cid]/total[cid]:8.3f}")


class          GT boxes top-1 acc
person             1408    0.685
bicycle              51    0.647
car                 258    0.655
motorcycle           69    0.870
bus                  45    0.911
truck                48    0.646
bench                61    0.492
bird                 67    0.537
dog                  21    0.905
horse                38    0.816
sheep                51    0.863
elephant             24    1.000
backpack             56    0.625
umbrella             43    0.744
bottle              127    0.843
chair               306    0.402
tv                   34    0.559
laptop               26    0.885
keyboard             22    0.455
book                184    0.685


Experiment B: RPN proposals + CLIP, fuse RPN objectness with CLIP via alpha.
Run on val-tune only (tuning split — val-test stays untouched for final evaluation).

In [4]:
%run scripts/run_clip.py --mode proposals --config configs/config.yaml \
    --checkpoint outputs/checkpoints/smoke/epoch_1.pt \
    --image-ids data/splits/val_tune_image_ids.json \
    --annotations data/annotations/instances_val_tune2017.stripped.json \
    --images-root data/images/val2017 \
    --proposals-dir outputs/proposals/val_tune --top-k 100 --alpha 0.0 \
    --out-dir outputs/clip/val_tune_prop_a0.0

%run scripts/run_clip.py --mode proposals --config configs/config.yaml \
    --checkpoint outputs/checkpoints/smoke/epoch_1.pt \
    --image-ids data/splits/val_tune_image_ids.json \
    --annotations data/annotations/instances_val_tune2017.stripped.json \
    --images-root data/images/val2017 \
    --proposals-dir outputs/proposals/val_tune --top-k 100 --alpha 0.5 \
    --out-dir outputs/clip/val_tune_prop_a0.5

%run scripts/run_clip.py --mode proposals --config configs/config.yaml \
    --checkpoint outputs/checkpoints/smoke/epoch_1.pt \
    --image-ids data/splits/val_tune_image_ids.json \
    --annotations data/annotations/instances_val_tune2017.stripped.json \
    --images-root data/images/val2017 \
    --proposals-dir outputs/proposals/val_tune --top-k 100 --alpha 1.0 \
    --out-dir outputs/clip/val_tune_prop_a1.0


loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
mode=proposals images=500 predictions=50000
saved -> outputs\clip\val_tune_prop_a0.0\predictions.json
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
mode=proposals images=500 predictions=50000
saved -> outputs\clip\val_tune_prop_a0.5\predictions.json
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
mode=proposals images=500 predictions=50000
saved -> outputs\clip\val_tune_prop_a1.0\predictions.json


In [8]:
import json
from pathlib import Path

sel = json.load(open("data/splits/selected_categories.json"))
base_names = [c["name"] for c in sel["base"]]
novel_names = [c["name"] for c in sel["novel"]]
n2o = {c["name"]: int(c["id"]) for c in sel["base"] + sel["novel"]}

from src.openworld.evaluation import evaluate_predictions
from pycocotools.coco import COCO

coco = COCO("data/annotations/instances_val_tune2017.stripped.json")
val_ids = json.load(open("data/splits/val_tune_image_ids.json"))

for a in ("a0.0", "a0.5", "a1.0"):
    p = Path(f"outputs/clip/val_tune_prop_{a}/predictions.json")
    if not p.exists():
        continue
    preds = json.load(open(p))
    m = evaluate_predictions(preds, coco, val_ids, base_names, novel_names, n2o)
    print(f"{a}: base={m['base_AP50']:.3f} novel={m['novel_AP50']:.3f} gen={m['generalized_AP50']:.3f}")


loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
a0.0: base=0.037 novel=0.023 gen=0.034
a0.5: base=0.038 novel=0.027 gen=0.035
a1.0: base=0.007 novel=0.000 gen=0.006


In [7]:
from src.openworld.evaluation import per_class_ap

# A prediction of class 1 overlaps a class 2 object.
# The correct score for class 1 in this example is 0.
predictions = {
    1: [{"bbox": [0, 0, 10, 10], "category_id": 1, "score": 0.9}]
}

ground_truth = {
    1: [{"bbox": [0, 0, 10, 10], "category_id": 2}],
    2: [{"bbox": [0, 0, 10, 10], "category_id": 1}]
}

print("Evaluation check:", per_class_ap(predictions, ground_truth, 1))

Evaluation check: 0.0


In [4]:
import importlib
from pathlib import Path
import src.openworld.evaluation as evaluation

evaluation = importlib.reload(evaluation)

print("File being used:", evaluation.__file__)

file_text = Path(evaluation.__file__).read_text(encoding="utf-8")
print("Replacement code found:", "for iid, gs in gt_by_image.items()" in file_text)

predictions = {
    1: [{"bbox": [0, 0, 10, 10], "category_id": 1, "score": 0.9}]
}
ground_truth = {
    1: [{"bbox": [0, 0, 10, 10], "category_id": 2}],
    2: [{"bbox": [0, 0, 10, 10], "category_id": 1}]
}

print("Evaluation check:", evaluation.per_class_ap(predictions, ground_truth, 1))

File being used: c:\Users\karki\Downloads\main_openworld\openworld\src\openworld\evaluation.py
Replacement code found: False
Evaluation check: 1.0


In [5]:
from pathlib import Path
import importlib
import src.openworld.evaluation as evaluation

path = Path(evaluation.__file__)
text = path.read_text(encoding="utf-8")

old = '        gts[iid] = [[float(v) for v in g["bbox"]] for g in gt_by_image.get(iid, [])]'

new = '''    gts = {
        iid: [
            [float(v) for v in g["bbox"]]
            for g in boxes
            if int(g["category_id"]) == cat_id
        ]
        for iid, boxes in gt_by_image.items()
    }'''

if text.count(old) != 1:
    raise RuntimeError(
        "Could not safely find the line to update. "
        "Stop here and send me this output."
    )

backup = path.with_suffix(".py.bak")
if not backup.exists():
    backup.write_text(text, encoding="utf-8")

path.write_text(text.replace(old, new), encoding="utf-8")

importlib.invalidate_caches()
evaluation = importlib.reload(evaluation)

predictions = {
    1: [{"bbox": [0, 0, 10, 10], "category_id": 1, "score": 0.9}]
}
ground_truth = {
    1: [{"bbox": [0, 0, 10, 10], "category_id": 2}],
    2: [{"bbox": [0, 0, 10, 10], "category_id": 1}]
}

print("Evaluation check:", evaluation.per_class_ap(predictions, ground_truth, 1))
print("Backup saved at:", backup)

Evaluation check: 0.0
Backup saved at: c:\Users\karki\Downloads\main_openworld\openworld\src\openworld\evaluation.py.bak
